# MSLG-SPA 2026 — Seq2Seq Translation · Google Colab

**Task:** MSL glosses ↔ Spanish &nbsp;|&nbsp; **Model:** mBART-large-50 + LoRA (r=16) &nbsp;|&nbsp; **Metric:** chrF / BLEU / METEOR

---

## Before you start

1. `Runtime → Change runtime type → T4 GPU`
2. Upload to `MyDrive/ML_projects/MSLG-SPA-2026/data/raw/`:
   ```
   MSLG_SPA_train.txt   ← TSV training file (490 pairs)
   test_mslg2spa.tsv    ← test file (add when received)
   test_spa2mslg.tsv    ← test file (add when received)
   ```
3. `Runtime → Run all`
4. Checkpoints sync to Drive every epoch — safe to disconnect and resume.


In [ ]:
# 1.1 — GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected!\n'
        'Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU'
    )

device   = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')
print(f'torch {torch.__version__}  |  CUDA {torch.version.cuda}')

In [ ]:
# 1.2 — Mount Drive + configure paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ================================================================
# CONFIGURE PATHS — edit only here
DRIVE_BASE = Path('/content/drive/MyDrive/ML_projects/MSLG-SPA-2026')
# ================================================================
DRIVE_DATA = DRIVE_BASE / 'data/raw'
DRIVE_CKPT = DRIVE_BASE / 'checkpoints'
DRIVE_SUB  = DRIVE_BASE / 'submissions'

PROJECT_ROOT = Path('/content/mslg-spa-2026')
LOCAL_DATA   = Path('/content/data_local')

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_SUB]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Drive base : {DRIVE_BASE}')
print(f'Drive data : {DRIVE_DATA}')
print(f'Drive ckpt : {DRIVE_CKPT}')

In [ ]:
# 1.3 — Install packages
!pip install -q 'peft>=0.10.0' evaluate sacrebleu nltk unbabel-comet tqdm
import nltk
nltk.download('wordnet',   quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Packages ready')

## 2 · Project Source Files
*Run once — collapse after first run.*


In [ ]:
# 2.0 — Directory structure
import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/mslg-spa-2026')
for d in ['src/data', 'src/models', 'src/evaluation',
          'utils', 'configs', 'scripts',
          'checkpoints/mslg2spa', 'checkpoints/spa2mslg', 'outputs']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

for pkg in ['src', 'src/data', 'src/models', 'src/evaluation', 'utils', 'scripts']:
    init = PROJECT_ROOT / pkg / '__init__.py'
    if not init.exists():
        init.write_text('')

print(f'Directory structure ready at {PROJECT_ROOT}')

In [ ]:
%%writefile /content/mslg-spa-2026/utils/seed.py
import random
import numpy as np
import torch


def set_global_seed(seed: int = 42) -> None:
    '''Set all random seeds for reproducibility.

    Args:
        seed: Integer seed value.
    '''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile /content/mslg-spa-2026/src/data/dataset.py
'''Dataset loading and preprocessing for MSLG-SPA 2026.'''

import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizer


def load_pairs(filepath) -> pd.DataFrame:
    '''Load a TSV file of (gloss, spanish) pairs into a DataFrame.

    Args:
        filepath: Path to a tab-separated file with header MSLG / SPA.

    Returns:
        DataFrame with columns [mslg, spa].
    '''
    df = pd.read_csv(filepath, sep='\t', header=0, names=['mslg', 'spa'])
    df = df.dropna()
    df['mslg'] = df['mslg'].str.strip()
    df['spa']  = df['spa'].str.strip()
    df = df[(df['mslg'] != '') & (df['spa'] != '')]
    df = df.reset_index(drop=True)
    return df


def print_stats(df: pd.DataFrame, name: str = 'Dataset') -> None:
    '''Print basic corpus statistics.'''
    line = '=' * 40
    print(f'\n{line}')
    print(f'  {name}')
    print(f'{line}')
    mslg_lens = df['mslg'].str.split().str.len()
    spa_lens  = df['spa'].str.split().str.len()
    print(f'  Pairs:             {len(df)}')
    print(f'  Avg MSLG tokens:   {mslg_lens.mean():.1f}')
    print(f'  Avg SPA tokens:    {spa_lens.mean():.1f}')
    print(f'  Max MSLG tokens:   {mslg_lens.max()}')
    print(f'  Max SPA tokens:    {spa_lens.max()}')


class TranslationDataset(Dataset):
    '''PyTorch Dataset for seq2seq translation (mslg2spa or spa2mslg).

    Args:
        data:        DataFrame with columns [mslg, spa].
        tokenizer:   HuggingFace tokenizer.
        subtask:     mslg2spa or spa2mslg.
        max_src_len: Max token length for source sequences.
        max_tgt_len: Max token length for target sequences.
    '''

    def __init__(
        self,
        data: pd.DataFrame,
        tokenizer: PreTrainedTokenizer,
        subtask: str,
        max_src_len: int = 128,
        max_tgt_len: int = 128,
    ) -> None:
        assert subtask in ('mslg2spa', 'spa2mslg'), \
            'subtask must be mslg2spa or spa2mslg'
        self.tokenizer   = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        if subtask == 'mslg2spa':
            self.sources = data['mslg'].tolist()
            self.targets = data['spa'].tolist()
        else:
            self.sources = data['spa'].tolist()
            self.targets = data['mslg'].tolist()

    def __len__(self) -> int:
        return len(self.sources)

    def __getitem__(self, idx: int) -> dict:
        source = self.sources[idx]
        target = self.targets[idx]
        model_inputs = self.tokenizer(
            source,
            max_length=self.max_src_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        labels = self.tokenizer(
            text_target=target,
            max_length=self.max_tgt_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        label_ids = labels['input_ids'].squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        return {
            'input_ids':      model_inputs['input_ids'].squeeze(),
            'attention_mask': model_inputs['attention_mask'].squeeze(),
            'labels':         label_ids,
        }

In [ ]:
%%writefile /content/mslg-spa-2026/src/models/seq2seq.py
'''Model loading utilities for MSLG-SPA 2026.

Uses mBART-large-50 with LoRA for parameter-efficient fine-tuning
on the low-resource MSL-Spanish parallel corpus.
'''

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


def load_model_and_tokenizer(
    model_name: str,
    use_lora: bool = True,
    lora_r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.1,
):
    '''Load a seq2seq model and tokenizer, optionally wrapping with LoRA.

    Args:
        model_name:   HuggingFace model identifier.
        use_lora:     Whether to apply LoRA (recommended for this task).
        lora_r:       LoRA rank.
        lora_alpha:   LoRA scaling factor (typically 2x r).
        lora_dropout: Dropout applied to LoRA layers.

    Returns:
        (model, tokenizer) tuple ready for training or inference.
    '''
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    if use_lora:
        from peft import LoraConfig, TaskType, get_peft_model

        lora_config = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=['q_proj', 'v_proj'],
            bias='none',
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    return model, tokenizer


def count_parameters(model) -> dict:
    '''Return total and trainable parameter counts.

    Returns:
        Dict with keys total and trainable.
    '''
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'total': total, 'trainable': trainable}

In [ ]:
%%writefile /content/mslg-spa-2026/src/evaluation/metrics.py
'''Metric computation for MSLG-SPA 2026.

Official metrics:
  - MSLG2SPA: BLEU, METEOR, chrF, COMET
  - SPA2MSLG: BLEU, METEOR, chrF  (no COMET)
'''

import evaluate
import numpy as np


def compute_bleu(predictions: list, references: list) -> float:
    '''Corpus-level BLEU score.

    Returns:
        BLEU score in [0, 100].
    '''
    metric = evaluate.load('sacrebleu')
    result = metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )
    return result['score']


def compute_chrf(predictions: list, references: list) -> float:
    '''Corpus-level chrF score.

    Returns:
        chrF score in [0, 100].
    '''
    metric = evaluate.load('chrf')
    result = metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )
    return result['score']


def compute_meteor(predictions: list, references: list) -> float:
    '''Corpus-level METEOR score.

    Returns:
        METEOR score in [0, 1].
    '''
    metric = evaluate.load('meteor')
    result = metric.compute(predictions=predictions, references=references)
    return result['meteor']


def compute_comet(
    sources: list,
    predictions: list,
    references: list,
) -> float:
    '''COMET score — only for MSLG2SPA subtask.

    Returns:
        COMET system-level score in roughly [-1, 1].
    '''
    try:
        from comet import download_model, load_from_checkpoint
        model_path  = download_model('Unbabel/wmt22-comet-da')
        comet_model = load_from_checkpoint(model_path)
        data = [
            {'src': s, 'mt': p, 'ref': r}
            for s, p, r in zip(sources, predictions, references)
        ]
        output = comet_model.predict(data, batch_size=8, gpus=1)
        return output['system_score']
    except ImportError:
        print('[WARNING] comet not installed. Skipping COMET.')
        return float('nan')


def evaluate_subtask(
    sources: list,
    predictions: list,
    references: list,
    subtask: str,
) -> dict:
    '''Compute all official metrics for a given subtask.

    Args:
        sources:     Source sentences.
        predictions: System outputs.
        references:  Gold references.
        subtask:     mslg2spa or spa2mslg.

    Returns:
        Dictionary with metric names as keys and scores as values.
    '''
    assert subtask in ('mslg2spa', 'spa2mslg')

    results = {}
    results['bleu']   = compute_bleu(predictions, references)
    results['chrf']   = compute_chrf(predictions, references)
    results['meteor'] = compute_meteor(predictions, references)

    if subtask == 'mslg2spa':
        results['comet'] = compute_comet(sources, predictions, references)

    line = '=' * 40
    print(f'\n{line}')
    print(f'  Results — {subtask.upper()}')
    print(f'{line}')
    for k, v in results.items():
        print(f'  {k.upper():<10}: {v:.4f}')

    return results


def compute_global_score(
    scores_per_system: list,
    subtask: str,
) -> list:
    '''Compute the official Global Score (mean of z-normalized metrics).

    Args:
        scores_per_system: List of metric dicts, one per system.
        subtask:           Determines which metrics to include.

    Returns:
        List of Global Scores, one per system.
    '''
    metrics = ['bleu', 'chrf', 'meteor']
    if subtask == 'mslg2spa':
        metrics.append('comet')

    matrix = np.array(
        [[s[m] for m in metrics] for s in scores_per_system],
        dtype=float,
    )
    means = matrix.mean(axis=0)
    stds  = matrix.std(axis=0)
    stds[stds == 0] = 1.0

    normalized    = (matrix - means) / stds
    global_scores = normalized.mean(axis=1).tolist()
    return global_scores

In [ ]:
%%writefile /content/mslg-spa-2026/configs/baseline.yaml
model:
  name: facebook/mbart-large-50
  max_source_length: 128
  max_target_length: 128

lora:
  enabled: true
  r: 16
  lora_alpha: 32
  lora_dropout: 0.1

data:
  train_file: data/raw/MSLG_SPA_train.txt
  test_mslg2spa: data/raw/test_mslg2spa.tsv
  test_spa2mslg: data/raw/test_spa2mslg.tsv
  processed_dir: data/processed/
  val_split: 0.15

training:
  output_dir: checkpoints/baseline
  num_train_epochs: 30
  per_device_train_batch_size: 8
  per_device_eval_batch_size: 16
  learning_rate: 5.0e-4
  warmup_steps: 50
  weight_decay: 0.01
  eval_strategy: epoch
  save_strategy: epoch
  load_best_model_at_end: true
  metric_for_best_model: chrf
  greater_is_better: true
  fp16: true
  seed: 42

generation:
  num_beams: 5
  max_new_tokens: 128

logging:
  report_to: none
  logging_steps: 10

In [ ]:
%%writefile /content/mslg-spa-2026/scripts/run_evaluate.py
'''Evaluation entry point for MSLG-SPA 2026.

Usage:
    python scripts/run_evaluate.py --config configs/baseline.yaml --subtask mslg2spa
    python scripts/run_evaluate.py --config configs/baseline.yaml --subtask spa2mslg
'''

import argparse
import json
import yaml
import torch
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM
from peft import PeftModel

from src.data.dataset import load_pairs
from src.evaluation.metrics import evaluate_subtask


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config',  required=True)
    parser.add_argument('--subtask', required=True, choices=['mslg2spa', 'spa2mslg'])
    return parser.parse_args()


def load_config(path: str) -> dict:
    with open(path, 'r') as f:
        return yaml.safe_load(f)


def load_trained_model(checkpoint_dir: str):
    from transformers import MBart50Tokenizer
    checkpoint_dir = Path(checkpoint_dir)

    tokenizer = MBart50Tokenizer.from_pretrained('facebook/mbart-large-50')

    if (checkpoint_dir / 'adapter_config.json').exists():
        adapter_config  = json.load(open(checkpoint_dir / 'adapter_config.json'))
        base_model_name = adapter_config['base_model_name_or_path']
        base_model = AutoModelForSeq2SeqLM.from_pretrained(
            base_model_name, local_files_only=False
        )
        model = PeftModel.from_pretrained(
            base_model, str(checkpoint_dir), local_files_only=False
        )
        model = model.merge_and_unload()
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            str(checkpoint_dir), local_files_only=False
        )
    model.eval()
    return model, tokenizer


def generate_translations(
    model,
    tokenizer,
    sources: list,
    subtask: str = 'mslg2spa',
    max_src_len: int = 128,
    max_new_tokens: int = 128,
    num_beams: int = 5,
) -> list:
    '''Generate translations for a list of source sentences.

    Args:
        model:          Trained seq2seq model.
        tokenizer:      Corresponding tokenizer.
        sources:        List of source sentences to translate.
        subtask:        mslg2spa or spa2mslg.
        max_src_len:    Max tokenization length for sources.
        max_new_tokens: Max tokens to generate per translation.
        num_beams:      Beam search width.

    Returns:
        List of translated strings.
    '''
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = model.to(device)

    # mBART requires forced_bos_token_id to select target language
    # Both subtasks use es_XX: Spanish for MSLG2SPA, and Spanish script for SPA2MSLG
    forced_bos_token_id = tokenizer.lang_code_to_id.get('es_XX')

    translations = []
    for source in sources:
        inputs = tokenizer(
            source,
            return_tensors='pt',
            max_length=max_src_len,
            truncation=True,
            padding=True,
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                early_stopping=True,
                forced_bos_token_id=forced_bos_token_id,
            )

        translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        translations.append(translation)

    return translations


def main():
    args   = parse_args()
    config = load_config(args.config)

    if args.subtask == 'mslg2spa':
        test_file          = config['data']['test_mslg2spa']
        src_col, tgt_col   = 'mslg', 'spa'
    else:
        test_file          = config['data']['test_spa2mslg']
        src_col, tgt_col   = 'spa', 'mslg'

    df         = load_pairs(test_file)
    sources    = df[src_col].tolist()
    references = df[tgt_col].tolist()
    print(f'Loaded {len(df)} test pairs for {args.subtask}')

    checkpoint_dir   = Path(config['training']['output_dir']) / 'final'
    model, tokenizer = load_trained_model(str(checkpoint_dir))
    print(f'Loaded model from {checkpoint_dir}')

    print('Generating translations...')
    predictions = generate_translations(
        model=model,
        tokenizer=tokenizer,
        sources=sources,
        subtask=args.subtask,
        max_src_len=config['model']['max_source_length'],
        max_new_tokens=config['generation']['max_new_tokens'],
        num_beams=config['generation']['num_beams'],
    )

    evaluate_subtask(
        sources=sources,
        predictions=predictions,
        references=references,
        subtask=args.subtask,
    )


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/mslg-spa-2026/scripts/predict.py
'''Generate official submission files for MSLG-SPA 2026.

Usage:
    python scripts/predict.py --config configs/baseline.yaml --subtask mslg2spa --team YourTeam --solution baseline
    python scripts/predict.py --config configs/baseline.yaml --subtask spa2mslg --team YourTeam --solution baseline
'''

import argparse
import yaml
from pathlib import Path

from src.data.dataset import load_pairs
from scripts.run_evaluate import load_trained_model, generate_translations


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config',   required=True)
    parser.add_argument('--subtask',  required=True, choices=['mslg2spa', 'spa2mslg'])
    parser.add_argument('--team',     required=True, help='Team name')
    parser.add_argument('--solution', required=True, help='Solution name (e.g. baseline)')
    return parser.parse_args()


def load_config(path: str) -> dict:
    with open(path, 'r') as f:
        return yaml.safe_load(f)


def write_submission(predictions: list, output_path: Path) -> None:
    '''Write predictions in the official submission format.

    Format: one prediction per line, wrapped in double quotes.
    '''
    with open(output_path, 'w', encoding='utf-8') as f:
        for pred in predictions:
            f.write(f'"{pred}"\n')
    print(f'Submission saved to {output_path}  ({len(predictions)} lines)')


def main():
    args   = parse_args()
    config = load_config(args.config)

    if args.subtask == 'mslg2spa':
        test_file = config['data']['test_mslg2spa']
        src_col   = 'mslg'
    else:
        test_file = config['data']['test_spa2mslg']
        src_col   = 'spa'

    df      = load_pairs(test_file)
    sources = df[src_col].tolist()
    print(f'Loaded {len(sources)} test instances for {args.subtask}')

    checkpoint_dir   = Path(config['training']['output_dir']) / 'final'
    model, tokenizer = load_trained_model(str(checkpoint_dir))

    print('Generating translations...')
    predictions = generate_translations(
        model=model,
        tokenizer=tokenizer,
        sources=sources,
        subtask=args.subtask,
        max_src_len=config['model']['max_source_length'],
        max_new_tokens=config['generation']['max_new_tokens'],
        num_beams=config['generation']['num_beams'],
    )

    filename    = f'{args.team}_{args.solution}_{args.subtask.upper()}.txt'
    output_path = Path('outputs') / filename
    output_path.parent.mkdir(exist_ok=True)
    write_submission(predictions, output_path)


if __name__ == '__main__':
    main()

## 3 · Data Setup


In [ ]:
# 3.1 — Load training data from Drive into RAM
import shutil
from pathlib import Path

LOCAL_DATA = Path('/content/data_local')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = LOCAL_DATA / 'MSLG_SPA_train.txt'

if TRAIN_FILE.exists():
    print('Data already in RAM — skipping.')
else:
    src = DRIVE_DATA / 'MSLG_SPA_train.txt'
    if not src.exists():
        raise FileNotFoundError(f'Training file not found on Drive: {src}')
    shutil.copy2(src, TRAIN_FILE)
    print(f'Copied MSLG_SPA_train.txt to {LOCAL_DATA}')

# Copy test files if present
for test_name in ['test_mslg2spa.tsv', 'test_spa2mslg.tsv']:
    src = DRIVE_DATA / test_name
    dst = LOCAL_DATA / test_name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f'Copied {test_name}')
    elif not src.exists():
        print(f'  (not yet on Drive: {test_name})')

In [ ]:
# 3.2 — Verify data integrity
import sys
from pathlib import Path

sys.path.insert(0, '/content/mslg-spa-2026')
from src.data.dataset import load_pairs, print_stats

df = load_pairs(LOCAL_DATA / 'MSLG_SPA_train.txt')
print_stats(df, name='MSLG_SPA_train')

print(f'\nSample pairs:')
for _, row in df.head(3).iterrows():
    print(f'  MSLG: {row["mslg"]}')
    print(f'  SPA:  {row["spa"]}')
    print()

## 4 · Restore Checkpoints from Drive

**Always run this cell before training.** On a fresh run it is a no-op.
On subsequent runs it restores any existing checkpoints from Drive into local directories,
enabling HuggingFace Trainer to resume automatically.


In [ ]:
# 4.0 — Restore checkpoints from Drive → local
import shutil
from pathlib import Path

restored = 0
for subtask in ['mslg2spa', 'spa2mslg']:
    drive_subtask = DRIVE_CKPT / subtask
    local_subtask = Path(f'/content/mslg-spa-2026/checkpoints/{subtask}')
    local_subtask.mkdir(parents=True, exist_ok=True)

    if not drive_subtask.exists():
        continue

    for ckpt in drive_subtask.iterdir():
        dest = local_subtask / ckpt.name
        if not dest.exists():
            if ckpt.is_dir():
                shutil.copytree(ckpt, dest)
            else:
                shutil.copy2(ckpt, dest)
            print(f'  Restored [{subtask}] {ckpt.name}')
            restored += 1

if restored == 0:
    print('No checkpoints on Drive — fresh training run.')
else:
    print(f'\nRestored {restored} checkpoint(s) from Drive.')

## 5 · Training


In [ ]:
# 5.1 — Training parameters
# ================================================================
# Edit here — everything else is read from configs/baseline.yaml
N_EPOCHS   = 30    # max epochs per subtask
BATCH_SIZE = 8     # T4 fits batch=8 with LoRA r=16
# ================================================================

import yaml, sys
sys.path.insert(0, '/content/mslg-spa-2026')

with open('/content/mslg-spa-2026/configs/baseline.yaml') as f:
    CONFIG = yaml.safe_load(f)

CONFIG['training']['num_train_epochs']          = N_EPOCHS
CONFIG['training']['per_device_train_batch_size'] = BATCH_SIZE
CONFIG['data']['train_file'] = str(LOCAL_DATA / 'MSLG_SPA_train.txt')

print(f'Model     : {CONFIG["model"]["name"]}')
print(f'Epochs    : {N_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {CONFIG["training"]["learning_rate"]}')
print(f'LoRA      : r={CONFIG["lora"]["r"]}, alpha={CONFIG["lora"]["lora_alpha"]}')
print(f'num_beams : {CONFIG["generation"]["num_beams"]}')

In [ ]:
# 5.2 — Train MSLG2SPA
import shutil, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
    TrainerCallback,
)
import evaluate as hf_evaluate

from utils.seed import set_global_seed
from src.data.dataset import load_pairs, print_stats, TranslationDataset
from src.models.seq2seq import load_model_and_tokenizer

SUBTASK            = 'mslg2spa'
OUTPUT_DIR         = Path('/content/mslg-spa-2026/checkpoints/mslg2spa')
DRIVE_CKPT_SUBTASK = DRIVE_CKPT / SUBTASK
DRIVE_CKPT_SUBTASK.mkdir(parents=True, exist_ok=True)

set_global_seed(CONFIG['training']['seed'])

# Drive backup callback — runs after each epoch save
class DriveBackupCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = Path(args.output_dir)
        checkpoints = sorted(
            [c for c in ckpt_dir.glob('checkpoint-*') if c.is_dir()],
            key=lambda x: int(x.name.split('-')[1]),
        )
        if checkpoints:
            latest = checkpoints[-1]
            dest = DRIVE_CKPT_SUBTASK / latest.name
            dest.mkdir(parents=True, exist_ok=True)
            for f in latest.iterdir():
                shutil.copy2(f, dest / f.name)
            print(f'  [Drive] Backed up {latest.name}')

# Data
df = load_pairs(CONFIG['data']['train_file'])
print_stats(df, name='Training data')
train_df, val_df = train_test_split(
    df,
    test_size=CONFIG['data']['val_split'],
    random_state=CONFIG['training']['seed'],
)
print(f'  Train: {len(train_df)}  |  Val: {len(val_df)}')

# Model
model, tokenizer = load_model_and_tokenizer(
    model_name=CONFIG['model']['name'],
    use_lora=CONFIG['lora']['enabled'],
    lora_r=CONFIG['lora']['r'],
    lora_alpha=CONFIG['lora']['lora_alpha'],
    lora_dropout=CONFIG['lora']['lora_dropout'],
)

# Datasets
train_dataset = TranslationDataset(
    train_df, tokenizer, SUBTASK,
    CONFIG['model']['max_source_length'],
    CONFIG['model']['max_target_length'],
)
val_dataset = TranslationDataset(
    val_df, tokenizer, SUBTASK,
    CONFIG['model']['max_source_length'],
    CONFIG['model']['max_target_length'],
)

# Compute metrics
chrf_metric = hf_evaluate.load('chrf')
bleu_metric = hf_evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds  = np.clip(preds, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    dec_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    dec_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    chrf = chrf_metric.compute(predictions=dec_preds, references=[[r] for r in dec_labels])
    bleu = bleu_metric.compute(predictions=dec_preds, references=[[r] for r in dec_labels])
    return {'chrf': chrf['score'], 'bleu': bleu['score']}

# Training args
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=CONFIG['training']['num_train_epochs'],
    per_device_train_batch_size=CONFIG['training']['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['training']['per_device_eval_batch_size'],
    learning_rate=CONFIG['training']['learning_rate'],
    warmup_steps=CONFIG['training']['warmup_steps'],
    weight_decay=CONFIG['training']['weight_decay'],
    eval_strategy=CONFIG['training']['eval_strategy'],
    save_strategy=CONFIG['training']['save_strategy'],
    load_best_model_at_end=CONFIG['training']['load_best_model_at_end'],
    metric_for_best_model=CONFIG['training']['metric_for_best_model'],
    greater_is_better=CONFIG['training']['greater_is_better'],
    predict_with_generate=True,
    fp16=True,
    seed=CONFIG['training']['seed'],
    report_to='none',
    logging_steps=CONFIG['logging']['logging_steps'],
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
    compute_metrics=compute_metrics,
    callbacks=[DriveBackupCallback()],
)

# Resume from checkpoint if available
resume_ckpt = None
if OUTPUT_DIR.exists():
    existing = sorted(
        [c for c in OUTPUT_DIR.glob('checkpoint-*') if c.is_dir()],
        key=lambda x: int(x.name.split('-')[1]),
    )
    if existing:
        resume_ckpt = str(existing[-1])
        print(f'Resuming from: {resume_ckpt}')

# Train
print(f'\nStarting training — subtask: {SUBTASK}')
trainer.train(resume_from_checkpoint=resume_ckpt)

# Save final model
final_dir = OUTPUT_DIR / 'final'
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
shutil.copytree(final_dir, DRIVE_CKPT_SUBTASK / 'final', dirs_exist_ok=True)
print(f'\nModel saved to Drive: {DRIVE_CKPT_SUBTASK}/final')

In [ ]:
# 5.3 — Train SPA2MSLG
# Same pipeline as 5.2 — only subtask and paths change
import shutil, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
    TrainerCallback,
)
import evaluate as hf_evaluate

from utils.seed import set_global_seed
from src.data.dataset import load_pairs, print_stats, TranslationDataset
from src.models.seq2seq import load_model_and_tokenizer

SUBTASK            = 'spa2mslg'
OUTPUT_DIR         = Path('/content/mslg-spa-2026/checkpoints/spa2mslg')
DRIVE_CKPT_SUBTASK = DRIVE_CKPT / SUBTASK
DRIVE_CKPT_SUBTASK.mkdir(parents=True, exist_ok=True)

set_global_seed(CONFIG['training']['seed'])

class DriveBackupCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = Path(args.output_dir)
        checkpoints = sorted(
            [c for c in ckpt_dir.glob('checkpoint-*') if c.is_dir()],
            key=lambda x: int(x.name.split('-')[1]),
        )
        if checkpoints:
            latest = checkpoints[-1]
            dest = DRIVE_CKPT_SUBTASK / latest.name
            dest.mkdir(parents=True, exist_ok=True)
            for f in latest.iterdir():
                shutil.copy2(f, dest / f.name)
            print(f'  [Drive] Backed up {latest.name}')

df = load_pairs(CONFIG['data']['train_file'])
train_df, val_df = train_test_split(
    df,
    test_size=CONFIG['data']['val_split'],
    random_state=CONFIG['training']['seed'],
)

model, tokenizer = load_model_and_tokenizer(
    model_name=CONFIG['model']['name'],
    use_lora=CONFIG['lora']['enabled'],
    lora_r=CONFIG['lora']['r'],
    lora_alpha=CONFIG['lora']['lora_alpha'],
    lora_dropout=CONFIG['lora']['lora_dropout'],
)

train_dataset = TranslationDataset(
    train_df, tokenizer, SUBTASK,
    CONFIG['model']['max_source_length'],
    CONFIG['model']['max_target_length'],
)
val_dataset = TranslationDataset(
    val_df, tokenizer, SUBTASK,
    CONFIG['model']['max_source_length'],
    CONFIG['model']['max_target_length'],
)

chrf_metric = hf_evaluate.load('chrf')
bleu_metric = hf_evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds  = np.clip(preds, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    dec_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    dec_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    chrf = chrf_metric.compute(predictions=dec_preds, references=[[r] for r in dec_labels])
    bleu = bleu_metric.compute(predictions=dec_preds, references=[[r] for r in dec_labels])
    return {'chrf': chrf['score'], 'bleu': bleu['score']}

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=CONFIG['training']['num_train_epochs'],
    per_device_train_batch_size=CONFIG['training']['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['training']['per_device_eval_batch_size'],
    learning_rate=CONFIG['training']['learning_rate'],
    warmup_steps=CONFIG['training']['warmup_steps'],
    weight_decay=CONFIG['training']['weight_decay'],
    eval_strategy=CONFIG['training']['eval_strategy'],
    save_strategy=CONFIG['training']['save_strategy'],
    load_best_model_at_end=CONFIG['training']['load_best_model_at_end'],
    metric_for_best_model=CONFIG['training']['metric_for_best_model'],
    greater_is_better=CONFIG['training']['greater_is_better'],
    predict_with_generate=True,
    fp16=True,
    seed=CONFIG['training']['seed'],
    report_to='none',
    logging_steps=CONFIG['logging']['logging_steps'],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
    compute_metrics=compute_metrics,
    callbacks=[DriveBackupCallback()],
)

resume_ckpt = None
if OUTPUT_DIR.exists():
    existing = sorted(
        [c for c in OUTPUT_DIR.glob('checkpoint-*') if c.is_dir()],
        key=lambda x: int(x.name.split('-')[1]),
    )
    if existing:
        resume_ckpt = str(existing[-1])
        print(f'Resuming from: {resume_ckpt}')

print(f'\nStarting training — subtask: {SUBTASK}')
trainer.train(resume_from_checkpoint=resume_ckpt)

final_dir = OUTPUT_DIR / 'final'
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
shutil.copytree(final_dir, DRIVE_CKPT_SUBTASK / 'final', dirs_exist_ok=True)
print(f'\nModel saved to Drive: {DRIVE_CKPT_SUBTASK}/final')

## 6 · Generate Submissions

Run after both subtasks are trained. Generates the official submission `.txt` files
and copies them to Drive.


In [ ]:
# 6.1 — Generate MSLG2SPA submission
import shutil, torch
from pathlib import Path
from datetime import datetime
from google.colab import files

from src.data.dataset import load_pairs
from scripts.run_evaluate import load_trained_model, generate_translations

# ================================================================
TEAM_NAME  = 'YourTeam'     # <─ edit
SOLUTION   = 'baseline'     # <─ edit
SUBTASK    = 'mslg2spa'
# ================================================================

test_file = LOCAL_DATA / 'test_mslg2spa.tsv'
if not test_file.exists():
    raise FileNotFoundError(f'Test file not found: {test_file}')

df      = load_pairs(test_file)
sources = df['mslg'].tolist()
print(f'Loaded {len(sources)} test instances')

checkpoint_dir   = Path('/content/mslg-spa-2026/checkpoints/mslg2spa/final')
model, tokenizer = load_trained_model(str(checkpoint_dir))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Generating translations...')
predictions = generate_translations(
    model=model,
    tokenizer=tokenizer,
    sources=sources,
    subtask=SUBTASK,
    max_src_len=CONFIG['model']['max_source_length'],
    max_new_tokens=CONFIG['generation']['max_new_tokens'],
    num_beams=CONFIG['generation']['num_beams'],
)

filename    = f'{TEAM_NAME}_{SOLUTION}_{SUBTASK.upper()}.txt'
output_path = Path('/content/mslg-spa-2026/outputs') / filename
output_path.parent.mkdir(exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    for pred in predictions:
        f.write(f'"{pred}"\n')

shutil.copy2(output_path, DRIVE_SUB / filename)
print(f'Submission saved to Drive: {DRIVE_SUB / filename}')
files.download(str(output_path))

In [ ]:
# 6.2 — Generate SPA2MSLG submission
import shutil, torch
from pathlib import Path
from google.colab import files

from src.data.dataset import load_pairs
from scripts.run_evaluate import load_trained_model, generate_translations

# ================================================================
TEAM_NAME = 'YourTeam'   # <─ edit
SOLUTION  = 'baseline'   # <─ edit
SUBTASK   = 'spa2mslg'
# ================================================================

test_file = LOCAL_DATA / 'test_spa2mslg.tsv'
if not test_file.exists():
    raise FileNotFoundError(f'Test file not found: {test_file}')

df      = load_pairs(test_file)
sources = df['spa'].tolist()
print(f'Loaded {len(sources)} test instances')

checkpoint_dir   = Path('/content/mslg-spa-2026/checkpoints/spa2mslg/final')
model, tokenizer = load_trained_model(str(checkpoint_dir))

print('Generating translations...')
predictions = generate_translations(
    model=model,
    tokenizer=tokenizer,
    sources=sources,
    subtask=SUBTASK,
    max_src_len=CONFIG['model']['max_source_length'],
    max_new_tokens=CONFIG['generation']['max_new_tokens'],
    num_beams=CONFIG['generation']['num_beams'],
)

filename    = f'{TEAM_NAME}_{SOLUTION}_{SUBTASK.upper()}.txt'
output_path = Path('/content/mslg-spa-2026/outputs') / filename
output_path.parent.mkdir(exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    for pred in predictions:
        f.write(f'"{pred}"\n')

shutil.copy2(output_path, DRIVE_SUB / filename)
print(f'Submission saved to Drive: {DRIVE_SUB / filename}')
files.download(str(output_path))